In [1]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import PoreNetworkPermeabilityModel

device = "cuda" if torch.cuda.is_available() else "cpu"
NETWORK_DIR = ROOT / "outputs" / "networks"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [2]:
EPOCHS = 100
LR = 1e-3
HIDDEN = 64
LAYERS = 3
CHECKPOINT_PATH = MODEL_DIR / "gnn_pnm_best.pth"


In [3]:
paths = sorted(NETWORK_DIR.glob("network_train_*.pt"))
if not paths:
    paths = sorted(NETWORK_DIR.glob("*.pt"))
print("network files:", len(paths))

def load_network(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

networks = [load_network(path) for path in paths]
first = networks[0]
node_in = first.node_attr.shape[1]
edge_in = first.edge_attr.shape[1]
print("node_in:", node_in, "edge_in:", edge_in)


network files: 20
node_in: 12 edge_in: 12


In [4]:
model = PoreNetworkPermeabilityModel(node_in=node_in, edge_in=edge_in, hidden=HIDDEN, layers=LAYERS, mu=1.0e-3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
print("parameters:", sum(p.numel() for p in model.parameters()))


parameters: 101249


In [5]:
def target_k_from_metadata(network):
    k = network.metadata.get("openpnm_k")
    if k is None:
        raise ValueError("No target k found. Add LBM/lab/OpenPNM target to network.metadata['openpnm_k'].")
    return torch.tensor([k["kx"], k["ky"], k["kz"]], dtype=torch.float32)

def network_loss(network):
    network = network.to(device)
    target_k = target_k_from_metadata(network).to(device).clamp_min(1e-30)
    pred_k, log_g = model(
        network.node_attr,
        network.edge_index,
        network.edge_attr,
        network.coords,
        network.domain_size,
        log_g_hp=network.log_g_hp,
    )
    loss = F.smooth_l1_loss(torch.log(pred_k.clamp_min(1e-30)), torch.log(target_k))
    return loss, pred_k.detach(), target_k.detach(), log_g.detach()


In [6]:
best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for network in tqdm(networks, desc=f"epoch {epoch}"):
        optimizer.zero_grad(set_to_none=True)
        loss, pred_k, target_k, _ = network_loss(network)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach().cpu())
    epoch_loss /= max(len(networks), 1)

    print(f"epoch={epoch} loss={epoch_loss:.6f}")
    print("last pred:", pred_k.cpu(), "target:", target_k.cpu())

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "node_in": node_in,
            "edge_in": edge_in,
            "hidden": HIDDEN,
            "layers": LAYERS,
            "epoch": epoch,
            "loss": epoch_loss,
        }, CHECKPOINT_PATH)
        print("saved:", CHECKPOINT_PATH)


epoch 1: 100%|██████████| 20/20 [00:01<00:00, 13.68it/s]


epoch=1 loss=1.622345
last pred: tensor([1.2129e-12, 4.7773e-12, 1.2916e-12]) target: tensor([3.2331e-13, 1.6201e-13, 1.6507e-13])
saved: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\gnn_pnm_best.pth


epoch 2: 100%|██████████| 20/20 [00:00<00:00, 21.98it/s]


epoch=2 loss=1.372653
last pred: tensor([3.8199e-13, 2.0462e-12, 4.2744e-13]) target: tensor([3.2331e-13, 1.6201e-13, 1.6507e-13])
saved: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\gnn_pnm_best.pth


epoch 3: 100%|██████████| 20/20 [00:00<00:00, 21.90it/s]


...

epoch=98 loss=0.470728
last pred: tensor([1.3477e-13, 8.0193e-13, 1.8293e-13]) target: tensor([3.2331e-13, 1.6201e-13, 1.6507e-13])


epoch 99: 100%|██████████| 20/20 [00:00<00:00, 22.59it/s]


epoch=99 loss=0.462628
last pred: tensor([1.2632e-13, 7.2321e-13, 1.7316e-13]) target: tensor([3.2331e-13, 1.6201e-13, 1.6507e-13])


epoch 100: 100%|██████████| 20/20 [00:00<00:00, 20.53it/s]

epoch=100 loss=0.462879
last pred: tensor([1.4818e-13, 8.7307e-13, 1.9128e-13]) target: tensor([3.2331e-13, 1.6201e-13, 1.6507e-13])
